# Loading FirmGuard Data into MongoDB

## What we are doing
We are loading device fleet data from a CSV file into a MongoDB cloud database (Atlas).
This data will serve as the foundation for FirmGuard AI's firmware risk intelligence platform.

## Steps
1. **Cell 1** — Install required libraries (`pymongo`, `dotenv`, `pandas`)
2. **Cell 2** — Connect to MongoDB Atlas using credentials from `.env` file
3. **Cell 3** — Load the CSV file into a Pandas DataFrame (100 rows, 13 columns)
4. **Cell 4** — Insert all records into a MongoDB collection called `devices`

## Why MongoDB?
FirmGuard deals with diverse device data across vendors (Siemens, Honeywell, Panasonic etc.).
MongoDB's flexible document model handles varying device attributes better than rigid SQL tables.

## Data
- **Source:** `firmguard_demo.csv`
- **Database:** `firmguard_db`
- **Collection:** `devices`
- **Records:** 100 device snapshots with 13 attributes each

_________________________________________________________________________________________________


## Cell 1 — Connect to MongoDB Atlas

We load our database credentials securely from the `.env` file (never hardcode passwords!).
Using these credentials, we establish a live connection to our MongoDB Atlas cloud cluster.

- `load_dotenv()` reads the `.env` file into memory
- `MongoClient` opens the connection to Atlas
- `db` is our handle to the `firmguard_db` database — all future reads/writes go through this

In [1]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()

uri = os.getenv("MONGODB_URI")
db_name = os.getenv("MONGODB_DATABASE")

client = MongoClient(uri)
db = client[db_name]

print(f"Connected to database: {db_name}")
print("Collections:", db.list_collection_names())

Connected to database: firmguard_db
Collections: ['firmguard_db']


## Cell 2 — Load CSV Data

We read the FirmGuard device data from a local CSV file into a Pandas DataFrame.

- `read_csv()` loads the file from the current folder (same directory as this notebook)
- `df.shape` confirms we have **100 rows and 13 columns**
- `df.head()` gives a quick preview of the device records

In [2]:
import pandas as pd

df = pd.read_csv("firmguard_demo.csv")

print(df.shape)
print(df.head())

(100, 13)
  device_id manufacturer    device_type fleet_risk_tier  \
0    DEV022      Siemens  SCADA_Gateway        Standard   
1    DEV029    Honeywell            HMI            High   
2    DEV008    Panasonic         Router        Standard   
3    DEV011      Siemens         Router        Standard   
4    DEV041    Schneider            HMI            High   

   days_since_last_patch  version_jump_magnitude  vendor_severity_score  \
0                   82.8                    1.84                      1   
1                   28.9                    0.52                      1   
2                   76.9                    2.83                      2   
3                   22.4                    0.39                      1   
4                   37.8                    0.48                      1   

   error_rate_per_hour  kernel_bootloader_touched  \
0                0.123                          0   
1                0.291                          0   
2                1.618   

## Cell 3 — Insert Data into MongoDB

We push all 100 device records from our DataFrame into a MongoDB collection called `devices`.

- `db["devices"]` creates (or connects to) the collection inside `firmguard_db`
- `to_dict(orient="records")` converts each DataFrame row into a Python dictionary (MongoDB document)
- `insert_many()` uploads all records in one batch operation

In [3]:
collection = db["devices"]

records = df.to_dict(orient="records")
result = collection.insert_many(records)

print(f"Inserted {len(result.inserted_ids)} records")
print("Collection:", db.list_collection_names())

Inserted 100 records
Collection: ['firmguard_db', 'devices']


## Cell 4 — Verify Insertion

We fetch a single record back from MongoDB to confirm the data was stored correctly.

- `find_one()` retrieves the first document from the `devices` collection
- A successful result means our data is live in the cloud database

In [4]:
sample = collection.find_one()
print("Sample record from MongoDB:")
print(sample)

Sample record from MongoDB:
{'_id': ObjectId('69b4fecc28bbf62a26e90431'), 'device_id': 'DEV022', 'manufacturer': 'Siemens', 'device_type': 'SCADA_Gateway', 'fleet_risk_tier': 'Standard', 'days_since_last_patch': 82.8, 'version_jump_magnitude': 1.84, 'vendor_severity_score': 1, 'error_rate_per_hour': 0.123, 'kernel_bootloader_touched': 0, 'cross_vendor_network_stress': 0.46, 'failure_probability': 0.511, 'deployment_outcome': 'ROLLBACK_REQUIRED', 'time_to_failure_hrs': 48.4}


## Cell 5 — Fetch All Data Back from MongoDB

We retrieve all records from the `devices` collection back into a Pandas DataFrame.

- `find({})` fetches every document (empty filter = no conditions)
- `pd.DataFrame(list(cursor))` converts the results back into a familiar tabular format
- `df.shape` confirms all **100 records** made the round trip successfully

In [5]:
cursor = collection.find({})
df_from_mongo = pd.DataFrame(list(cursor))

print(df_from_mongo.shape)
print(df_from_mongo.head(1))

(100, 14)
                        _id device_id manufacturer    device_type  \
0  69b4fecc28bbf62a26e90431    DEV022      Siemens  SCADA_Gateway   

  fleet_risk_tier  days_since_last_patch  version_jump_magnitude  \
0        Standard                   82.8                    1.84   

   vendor_severity_score  error_rate_per_hour  kernel_bootloader_touched  \
0                      1                0.123                          0   

   cross_vendor_network_stress  failure_probability deployment_outcome  \
0                         0.46                0.511  ROLLBACK_REQUIRED   

   time_to_failure_hrs  
0                 48.4  
